In [1]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm

In [6]:
# Paths
root_dir = r"C:\Users\patel\Downloads\Lane_detection\bdd100k"
image_dir = os.path.join(root_dir, "images", "100k", "train")
label_dir = os.path.join(root_dir, "labels", "100k", "train")
output_vis_dir = os.path.join(root_dir, "processed", "visualization", "train")

In [7]:
os.makedirs(output_vis_dir, exist_ok=True)

In [8]:
# Colors (BGR)
LANE_COLOR = (255, 255, 255)     # white
DRIVABLE_COLOR = (128, 128, 128) # gray
CAR_COLOR = (0, 0, 255)          # red

In [9]:
for file in tqdm(os.listdir(label_dir)):
    if not file.endswith(".json"):
        continue

    label_path = os.path.join(label_dir, file)
    with open(label_path, "r") as f:
        data = json.load(f)

    img_name = data["name"] + ".jpg"
    img_path = os.path.join(image_dir, img_name)
    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]
    overlay = img.copy()

    # Loop through frames and objects
    for frame in data["frames"]:
        for obj in frame["objects"]:
            cat = obj["category"]

            # 1️⃣ Lane lines
            if "poly2d" in obj and cat.startswith("lane/"):
                pts = np.array([[p[0], p[1]] for p in obj["poly2d"]], np.int32)
                pts = pts.reshape((-1, 1, 2))
                cv2.polylines(overlay, [pts], isClosed=False, color=LANE_COLOR, thickness=3)

            # 2️⃣ Drivable area
            elif "poly2d" in obj and "drivable area" in cat.lower():
                pts = np.array([[p[0], p[1]] for p in obj["poly2d"]], np.int32)
                pts = pts.reshape((-1, 1, 2))
                cv2.fillPoly(overlay, [pts], DRIVABLE_COLOR)

            # 3️⃣ Cars / vehicles
            elif "box2d" in obj and any(v in cat.lower() for v in ["car", "truck", "bus", "vehicle"]):
                box = obj["box2d"]
                x1, y1 = int(box["x1"]), int(box["y1"])
                x2, y2 = int(box["x2"]), int(box["y2"])
                cv2.rectangle(overlay, (x1, y1), (x2, y2), CAR_COLOR, 2)

    # Resize for visualization (optional)
    overlay_resized = cv2.resize(overlay, (512, 256))
    # Save
    cv2.imwrite(os.path.join(output_vis_dir, file.replace(".json", ".jpg")), overlay_resized)

100%|████████████████████████████████████████████████████████████████████████████| 70000/70000 [19:54<00:00, 58.58it/s]


In [10]:
print("✅ Overlay visualization complete! Check:", output_vis_dir)

✅ Overlay visualization complete! Check: C:\Users\patel\Downloads\Lane_detection\bdd100k\processed\visualization\train
